# 🩺 Bangla Prescription Analyzer — Training in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/bangla-prescription-analyzer/blob/main/notebooks/train_colab.ipynb)

**Pipeline:** handwritten medicine-word image → CNN (78 medicine classes) → generic name → Bangla description (uses, side effects, cautions).

Dataset: [Doctor's Handwritten Prescription BD dataset](https://www.kaggle.com/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset) — 4,680 word images, 78 medicines, 60/20/20 split.

Run the cells top to bottom. Expected time on a free T4 GPU: ~15–25 min for EfficientNet-B0.

## Step 1 — Use a GPU runtime
Menu: **Runtime → Change runtime type → T4 GPU → Save**. Then run the cell below; you should see a Tesla T4.

In [ ]:
!nvidia-smi

## Step 2 — Mount Google Drive
Checkpoints are saved to Drive so they survive when Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = '/content/drive/MyDrive/rx_runs/effb0'   # change the name for each experiment
!mkdir -p $RUN_DIR

## Step 3 — Get the project code
**Option A (recommended):** after you push this repo to GitHub, clone it here (public repo).
**Option B:** private repo → use a GitHub token, or upload the zip with the file browser and unzip.

In [ ]:
REPO_URL = 'https://github.com/YOUR_GITHUB_USERNAME/bangla-prescription-analyzer.git'  # <-- edit

import os
if not os.path.exists('/content/bangla-prescription-analyzer'):
    !git clone $REPO_URL /content/bangla-prescription-analyzer
%cd /content/bangla-prescription-analyzer
!git pull

In [ ]:
# Colab already ships torch/torchvision/pandas/sklearn — install only what is missing
!pip install -q timm gradio kagglehub
import torch, timm; print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())

## Step 4 — Download the dataset from Kaggle
`kagglehub` downloads public datasets directly. If it asks for credentials: Kaggle → Settings → **API → Create New Token**, then add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's 🔑 **Secrets** panel (left sidebar) and enable notebook access.

In [ ]:
import os, kagglehub
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass  # public dataset usually works without a token

DATA_ROOT = kagglehub.dataset_download('mamun1113/doctors-handwritten-prescription-bd-dataset')
print('Dataset at:', DATA_ROOT)
!find "$DATA_ROOT" -maxdepth 3 -type d
!find "$DATA_ROOT" -name '*.csv'

## Step 5 — Explore the data (EDA)

In [ ]:
import sys; sys.path.insert(0, '.')
import pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from src.data import load_all

frames = load_all(DATA_ROOT)
for split, df in frames.items():
    print(f"{split:5s} {len(df):5d} images | {df['medicine'].nunique()} medicines | {df['generic'].nunique()} generics")

train = frames['train']
display(train.head())
print('\nImages per class (train):'); print(train['medicine'].value_counts().describe())
print('\nMedicines per generic:'); display(train.drop_duplicates('medicine').groupby('generic')['medicine'].apply(list))

In [ ]:
sample = train.sample(16, random_state=0)
fig, axes = plt.subplots(4, 4, figsize=(12, 8))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(r.path).convert('L'), cmap='gray'); ax.set_title(f"{r.medicine}\n({r.generic})", fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

sizes = pd.DataFrame([Image.open(p).size for p in train.path.sample(300, random_state=0)], columns=['w', 'h'])
print(sizes.describe())

## Step 6 — Train
Key choices (see `src/train.py`): ImageNet-pretrained backbone, AdamW, warm-up + cosine LR, label smoothing, mixed precision, early stopping, light augmentation (small rotation/shear/blur — **no flips**, because flipped text is a different word).

Try other backbones later by changing `--model`: `convnext_tiny`, `resnet50`, `mobilenetv3_large_100`, `efficientnet_b3`.

In [ ]:
!python -m src.train --data_root "$DATA_ROOT" --out_dir "$RUN_DIR" \
    --model efficientnet_b0 --img_size 224 --epochs 40 --batch_size 64 --lr 3e-4

In [ ]:
import json
h = pd.DataFrame(json.load(open(f'{RUN_DIR}/history.json')))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
h.plot(x='epoch', y=['train_loss', 'val_loss'], ax=ax[0], title='Loss')
h.plot(x='epoch', y=['train_acc', 'val_top1', 'val_top5'], ax=ax[1], title='Accuracy')
plt.show(); h.tail()

## Step 7 — Evaluate on the held-out Testing split
Reports top-1 / top-5 accuracy, macro-F1, and **generic accuracy** (a wrong brand with the same active ingredient is a less dangerous mistake).

In [ ]:
!python -m src.evaluate --data_root "$DATA_ROOT" --run_dir "$RUN_DIR"
from IPython.display import Image as IPImage
IPImage(f'{RUN_DIR}/confusion_matrix.png', width=900)

In [ ]:
wrong = pd.read_csv(f'{RUN_DIR}/misclassified.csv')
print(len(wrong), 'mistakes'); display(wrong.groupby(['medicine', 'pred']).size().sort_values(ascending=False).head(15))

## Step 8 — Bangla knowledge base
Check which generic names from the dataset still need a Bangla entry in `data/bangla_generic_info.json`.

In [ ]:
!python -m src.knowledge_base check --label_map "$RUN_DIR/label_map.json"

### (Optional) Enrich from a large Bangladesh medicine database
Downloads `generic.csv` from the MIT-licensed *Medicine Dataset of Bangladesh* on Hugging Face and builds `data/kb_extra.json` (English descriptions; add `--translate` for machine translation EN→BN with BanglaT5, slower and must be reviewed).

In [ ]:
from huggingface_hub import hf_hub_download
gcsv = hf_hub_download('ArnobBot/Medicine-Dataset-of-Bangladesh', 'generic.csv', repo_type='dataset')
print(pd.read_csv(gcsv, nrows=2).columns.tolist())
!python -m src.knowledge_base build --generic_csv "$gcsv" --label_map "$RUN_DIR/label_map.json"
# with translation:  !pip -q install transformers sentencepiece && python -m src.knowledge_base build ... --translate

## Step 9 — Analyze images (Bangla report)

In [ ]:
from IPython.display import Markdown, display
from src.analyzer import PrescriptionAnalyzer
an = PrescriptionAnalyzer(RUN_DIR)
test = frames['test'].sample(3, random_state=1)
for _, r in test.iterrows():
    display(Image.open(r.path).convert('L').resize((256, 128)))
    print('Ground truth:', r.medicine)
    display(Markdown(an.to_bangla_report(an.analyze(r.path))))

## Step 10 — Web demo (Gradio, public link)

In [ ]:
!python app/app.py --run_dir "$RUN_DIR" --share

## Step 11 — Save results back to GitHub (manually)
* **Do not commit `best.pt`/`last.pt`** (they are ignored by `.gitignore`). Keep them in Drive, or upload to a GitHub Release / Hugging Face Hub.
* Copy small result files you want in the repo into `results/` and download them:

In [ ]:
!mkdir -p results && cp "$RUN_DIR"/{config.json,history.json,test_metrics.json,classification_report.csv,confusion_matrix.png} results/ 2>/dev/null; ls results
!zip -qr /content/results.zip results && echo 'Download /content/results.zip from the file browser, unzip into your local repo, then commit & push.'